# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


## 06 — A auditoria oficial encontra os mesmos padrões?

### Pergunta
A auditoria oficial encontra os mesmos padrões?

### Motivação
Uma implementação independente pode revelar diferenças de definição, escalonamento e critérios de alerta.

### Método
Construir InputData com DataFrameInputDataBuilder; instanciar Meridian com defaults provisórios e MeridianEDA; gerar HTML, checks, warnings e artefatos numéricos. Recalcular R² ajustado na escala oficial por soma de quadrados e comparar com tolerância 1e-5.

### Interpretação antes de olhar os resultados
A API 2.1.0 gera amostras PRIOR automaticamente; não há posterior ou MCMC. Os defaults não são especificação final. INFO não significa ausência de problema; REVIEW e FAIL precisam ser lidos. R² oficial é ajustado e usa variáveis transformadas.

**Dependências:** painel validado e funções em `src/official_eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [2]:
from src.official_eda import official_eda
report_06 = official_eda(geo_time_panel)
show(report_06)

I0000 00:00:1791024369.423881    2516 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791024369.424314    2516 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1791024370.322845    2516 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791024370.323101    2516 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Resultado observado e conclusão parcial

O relatório HTML oficial foi executado com Meridian 2.1.0, sem amostragem posterior. Os checks exportaram 16 achados: {'INFO': 10, 'REVIEW': 6}. As advertências de execução estão em meridian_warnings.csv; não foram silenciadas nem resolvidas por alteração de thresholds.

A API atual usa DataFrameInputDataBuilder. MeridianEDA gera amostras da prior no construtor para completar diagnósticos da especificação. Isso não ajusta o modelo aos dados e não produz estimativas posteriores. A especificação default é provisória: 156 knots e 8 lags. Não escolhemos hiperparâmetros finais com esta EDA.

A razão oficial é 6240/(40−1+156+2+7) = 30.59. Essa contagem é um guardrail que não conta todos os efeitos geo como parâmetros independentes. Não é prova de tamanho amostral efetivo ou de suficiência.

Os alertas de CPMU precisam de contexto: o CV máximo do custo unitário é 3.475e-08. A variação relativa minúscula é compatível com arredondamento na geração/exportação, e não evidencia dispersão econômica de custos. Não alteramos o dado ou o threshold IQR oficial para eliminar o alerta.

Uma diferença metodológica importante: os R² oficiais são ajustados e usam dados transformados; a EDA própria mostra R² brutos e por habitante não ajustados. Recalculando por soma de quadrados na mesma escala oficial, a concordância dentro de 1e-5 foi **True**. A tabela official_r2_reconciliation contém as diferenças por variável. VIF e correlações também dependem da população e agregação. Artefatos numéricos foram exportados para investigar essas diferenças.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Consolidar os achados, implicações e limites de prontidão.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**official_checks** — 16 linhas; CSV completo em `outputs/tables/`.

,check,severity,cause,explanation
0,check_geo_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
1,check_national_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
2,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled KPI in certai...
3,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
4,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled KPI. Please c...
5,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
6,check_overall_kpi_invariability,INFO,NONE,The kpi_scaled has variability across geos and...
7,check_population_corr_raw_media,INFO,NONE,Please review the Spearman correlation between...
8,check_population_corr_scaled_treatment_control,INFO,NONE,Please review the Spearman correlation between...
9,check_geo_pairwise_corr,INFO,NONE,Please review the computed pairwise correlatio...


**data_adequacy** — 1 linhas; CSV completo em `outputs/tables/`.

,n_geos,n_times,n_knots,n_controls,n_treatments,n_data_points,n_parameters,ratio
0,40,156,156,2,7,6240,204,30.588235


**comparison** — 7 linhas; CSV completo em `outputs/tables/`.

,issue,custom_eda,custom_evidence,meridian_eda,agreement,interpretation
0,cost_per_media_unit,CPMU = spend/media; NaN em zero,cost_per_media_unit.csv,{'REVIEW': 2},"REVIEW por extremos, mas CV máximo de CPMU=3.4...",Mesmas unidades; verificar artefatos por geo e...
1,std,"Variância bruta, zeros, IQR e MAD",channel_summary.csv,{'REVIEW': 4},comparação de definições; não igualdade numéri...,Escalas distintas: Meridian transforma as vari...
2,population,Spearman de população × total de mídia,population_media_correlation.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Total e média temporal são proporcionais em pa...
3,pairwise,"Pearson/Spearman nacional, overall e within",correlation_summary.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Meridian usa transformação própria; within da ...
4,vif,VIF com intercepto em preditores brutos/within,vif_summary.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Diagnósticos oficiais usam escalas/população; ...
5,geo_time,R² não ajustado em valores brutos e per capita,geo_time_r2.csv,{'INFO': 2},R² ajustado reconciliado na mesma escala: True...,Meridian usa R² ajustado sobre variáveis trans...
6,data_param,G×T observações; parâmetros não são todos inde...,data_audit.csv,{'INFO': 1},comparação de definições; não igualdade numéri...,Guardrail provisório; razão não prova adequaçã...


In [3]:
from IPython.display import display, FileLink
display(FileLink("outputs/reports/meridian_eda_report.html"))
# Baixe o HTML e abra no navegador para explorar os gráficos oficiais.

/workspace/scratch/5d823022ed95/tcc-meridian/outputs/reports/meridian_eda_report.html